In [ ]:
import time
import boto3
import sagemaker
import json
import numpy as np
from sagemaker import get_execution_role
from sagemaker.amazon.amazon_estimator import get_image_uri
from sagemaker.tuner import IntegerParameter, CategoricalParameter, ContinuousParameter, HyperparameterTuner
from sagemaker.model import Model
from sagemaker.predictor import RealTimePredictor

# Configuring the built-in Image Classification algorithm

## Create a unique job name 

In [ ]:
job_name_prefix = 'breast-cancer-detection'
timestamp = time.strftime('-%Y-%m-%d-%H-%M-%S', time.gmtime())
job_name = job_name_prefix + timestamp

## Specify the input and output paths for the job

In [ ]:
bucket = 'alexbc.datalake.us-east-2'
input_prefix = 'breast-cancer-histopathology/curated-data/recordio'
input_train = 's3://{}/{}/train/'.format(bucket, input_prefix)
input_test = 's3://{}/{}/test/'.format(bucket, input_prefix)

output_prefix = 'breast-cancer-histopathology/model-output/sagemaker-api'
output_path = 's3://{}/{}/'.format(bucket, output_prefix)

## Creating an Estimator to build image classification

In [ ]:
role = get_execution_role()
training_image = get_image_uri(boto3.Session().region_name, 'image-classification')

sagemaker_session = sagemaker.Session()
estimator = sagemaker.estimator.Estimator(training_image, 
                                          role, 
                                          train_instance_count=1,
                                          train_instance_type='ml.p2.xlarge',
                                          train_volume_size=50,
                                          train_max_run=360000,
                                          output_path=output_path, 
                                          sagemaker_session=sagemaker_session,
                                          input_mode='Pipe')

In [ ]:
estimator.set_hyperparameters(num_classes=2,
                              num_training_samples=194266,
                              num_layers=18,
                              mini_batch_size=128,
                              image_shape='3,50,50',
                              augmentation_type='crop_color_transform',
                              epochs=5,
                              learning_rate=0.01,
                              use_pretrained_model=1)

# Create a training job

In [ ]:
s3_input_train = sagemaker.s3_input(s3_data=input_train, content_type='application/x-recordio')
s3_input_validation = sagemaker.s3_input(s3_data=input_test, content_type='application/x-recordio')

In [ ]:
estimator.fit({
    'train': s3_input_train,
    'validation': s3_input_validation
}, job_name=job_name)

# Deploying the best model found by the tuning job

## Get the execution role and the hosting image URI for Image Classification

In [ ]:
role = get_execution_role()
hosting_image = get_image_uri(boto3.Session().region_name, 'image-classification')

## Configure hosting instances

In [ ]:
instance_count = 1
instance_type = 'ml.m4.xlarge'

## Create a unique model name

In [ ]:
model_name_prefix = 'bcd-image-classification-high-level'
timestamp = time.strftime('-%Y-%m-%d-%H-%M-%S', time.gmtime())
model_name = model_name_prefix + timestamp

## Create a Model object

In [ ]:
model_artifacts_s3_path = 's3://sagemaker-data-jv/breast-cancer-detection/output/bcd-tuning-2019-06-17-12-24-43-001-f302e721/output/model.tar.gz'
model = Model(
    name=model_name,
    model_data=model_artifacts_s3_path,
    image=hosting_image,
    role=role,
    predictor_cls=lambda endpoint_name, sagemaker_session: RealTimePredictor(endpoint_name, sagemaker_session)
)

## Create a unique endpoint name

In [ ]:
endpoint_name_prefix = 'breast-cancer-detection-ep'
timestamp = time.strftime('-%Y-%m-%d-%H-%M-%S', time.gmtime())
endpoint_name = endpoint_name_prefix + timestamp

## Create a model, an endpoint configuration and an endpoint

In [ ]:
predictor = model.deploy(
    endpoint_name=endpoint_name,
    initial_instance_count=instance_count,
    instance_type=instance_type
)

# Testing the deployed model

In [ ]:
def predict_breast_cancer(image_path):
    with open(image_path, 'rb') as f:
        payload = f.read()
        payload = bytearray(payload)
    response = predictor.predict(payload)
    result = json.loads(response)
    print('Probabilities for all classes: ', result)
    predicted_class = np.argmax(result)
    if predicted_class == 0:
        print('Breast cancer not detected')
    else:
        print('Breast cancer detected')

In [ ]:
predict_breast_cancer('images/0/8975_idx5_x2851_y1201_class0.png')

In [ ]:
predict_breast_cancer('images/1/10253_idx5_x551_y651_class1.png')

# Deleting endpoint

In [ ]:
sagemaker.Session().delete_endpoint(predictor.endpoint)